# 4. Additional Graph

ROI 기반 좌표계 + 행동 상태 분류 파이프라인

| 코호트 | 신호 |
|--------|------|
| FIP_pdyn_FGCaMP | 470nm → Ca²⁺ (FGCaMP) |
| FIP_wt_CamKII+FGCaMP | 470nm → Ca²⁺ (FGCaMP) |
| FIP_wt_dLight+jRGECO | 470nm → DA (dLight), 565nm → Ca²⁺ (jRGECO) |

**행동 기준 (BEHAV)**
| 상태 | 조건 |
|------|------|
| Rest | 속도 < 2 cm/s |
| Movement | 속도 > 3 cm/s |
| Shelter stay | shelter polygon 내부 × 1 s 이상 (polygon 없을 시: 중심 8 cm 이내) |

**Geometry**: roi_info.json 에서 px_per_cm, shelter_center, arena_center, shelter_points 직접 로드

**사용법**: Cell 1 → 3 순서대로 실행 후 세션 선택 → Cell 4 → 10 실행

**Kernel**: nbi(python) 선택

## Cell 1 — 라이브러리 Import
`nbi` 커널 필요 (numpy, pandas, matplotlib, scipy, tkinter)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter1d
from scipy.interpolate import interp1d
from matplotlib.path import Path as MplPath
import tkinter as tk; from tkinter import filedialog
import os, glob, json as _json, warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.size': 8, 'font.family': 'Arial',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print('Setup complete')

## Cell 2 — 코호트 설정 (COHORT_CONFIG + BEHAV)
새 실험 추가 시 `COHORT_CONFIG`에 항목 삽입. 행동 기준값은 `BEHAV` dict 한 곳에서 관리.

In [ ]:
COHORT_CONFIG = {
    'FIP_pdyn_FGCaMP': {
        'description': 'Prodynorphin-Cre x FGCaMP  (D1-MSN, Ca2+)',
        'signals': {
            'ca': {'col': '470nm_dFF(%)', 'scale': 1/100,
                   'label': 'Ca2+ (FGCaMP)', 'color': '#2ca02c', 'cmap': 'RdBu_r'},
        },
        'primary': 'ca',
        'figures': ['fig3e', 'exfig2d', 'exfig2e'],
    },
    'FIP_wt_CamKII+FGCaMP': {
        'description': 'WT x CamKII-FGCaMP  (pan-neuron, Ca2+)',
        'signals': {
            'ca': {'col': '470nm_dFF(%)', 'scale': 1/100,
       'label': 'Ca2+ (FGCaMP)', 'color': '#2ca02c', 'cmap': 'RdBu_r'},
        },
        'primary': 'ca',
        'figures': ['fig3e', 'exfig2d', 'exfig2e'],
    },
    'FIP_wt_dLight+jRGECO': {
        'description': 'WT x dLight1.1 (DA) + jRGECO (Ca2+)',
        'signals': {
            'da': {'col': '470nm_dFF(%)', 'scale': 1/100,
       'label': 'DA (dLight1.1)', 'color': '#2ca02c', 'cmap': 'PuOr'},
            'ca': {'col': '565nm_dFF(%)', 'scale': 1/100,
                   'label': 'Ca2+ (jRGECO)', 'color': '#d62728', 'cmap': 'RdBu_r'},
        },
        'primary': 'da',
        'figures': ['fig3e', 'exfig2d', 'exfig2e', 'exfig3g'],
    },
}

# 행동 기준값 — 수정 시 여기만 변경
BEHAV = dict(
    rest_thr    = 2.0,   # cm/s 미만  → Rest
    move_thr    = 3.0,   # cm/s 초과  → Movement
    shelter_r   = 8.0,   # cm 이내    → Shelter stay (polygon 없을 시 fallback)
    shelter_dur = 1.0,   # s 이상 지속
    # genuine shelter entry: 직전에 명확히 밖에 있었어야 함
    genuine_entry_min_dist = 15.0,  # cm: entry 전 lookback 구간에서 최대 dist 기준
    genuine_entry_lookback =  5.0,  # s:  lookback 구간 길이
)
# ── 세션 구간 설정 ────────────────────────────────────────────────────────
# behavior/session_timestamps.csv 형식:
#   event,timestamp_s,local_time,duration_since_start_s
#   start,30101.905057,2026-07-16 08:21:41.905,
#   save ,31928.416903,2026-07-16 08:52:08.416,1826.512
# True → 세션 start ~ save 구간의 데이터만 남겨 모든 그래프를 그림
USE_SESSION_WINDOW   = True
SESSION_START_EVENTS = ('start',)                # 세션 시작 이벤트 이름
SESSION_END_EVENTS   = ('save', 'end', 'stop')   # 세션 종료 이벤트 이름

print('Config OK:', list(COHORT_CONFIG.keys()))
print('BEHAV:', BEHAV)
print('세션 구간 crop:', USE_SESSION_WINDOW)

## Cell 3 — 세션 & 저장경로 선택
폴더 선택 다이얼로그가 팝업됩니다. **취소**를 누르면 선택 완료. 저장경로 미선택 시 `D:\Data\abstractDATA_KSH` 기본값 사용.

In [ ]:
root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)

class _MultiDir:
    def __init__(self): self.dirs = []
    def pick(self):
        while True:
            d = filedialog.askdirectory(
                title=f'세션 폴더 선택 ({len(self.dirs)}개 선택됨) — 취소하면 완료',
                parent=root)
            if not d: break
            self.dirs.append(d)
        return self.dirs

selected_sessions = _MultiDir().pick()
if not selected_sessions:
    print('경고: 세션이 선택되지 않았습니다.')
else:
    print(f'{len(selected_sessions)}개 세션:')
    for p in selected_sessions: print('  ', p)

SAVE_DIR = filedialog.askdirectory(title='그림 저장 폴더 선택', parent=root)
if not SAVE_DIR:
    SAVE_DIR = r'D:\Data\abstractDATA_KSH'
    print('기본 저장경로:', SAVE_DIR)
else:
    print('저장경로:', SAVE_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 파장 선택 + 이름 입력 다이얼로그 ─────────────────────────────────────
def ask_signals():
    # (csv 컬럼명, 표시 라벨, 기본 체크 여부, 기본 신호 이름, 색상, cmap)
    WAVE_OPTIONS = [
        ('470nm_dFF(%)', '470 nm', True,  'Ca2+ (FGCaMP)',    '#2ca02c', 'PuOr'),
        ('565nm_dFF(%)', '565 nm', False, 'Ca2+ (jRGECO)',    '#d62728', 'RdBu_r'),
    ]

    win = tk.Toplevel(root)
    win.title('신호 설정')
    win.attributes('-topmost', True)
    win.grab_set()
    win.resizable(False, False)

    tk.Label(win, text='측정 파장 및 신호 이름 설정',
             font=('Arial', 11, 'bold')).grid(
        row=0, column=0, columnspan=3, pady=(16, 4), padx=28)
    tk.Label(win, text='(복수 선택 가능 / 이름은 자유롭게 수정)',
             font=('Arial', 9), fg='gray').grid(
        row=1, column=0, columnspan=3, pady=(0, 10))

    checks  = {}
    entries = {}
    for i, (col, wave_label, default_check, default_name, *_) in enumerate(WAVE_OPTIONS):
        var = tk.BooleanVar(value=default_check)
        checks[col] = var
        tk.Checkbutton(win, text=wave_label, variable=var,
                       font=('Arial', 10), width=8).grid(
            row=i+2, column=0, padx=(28, 4), pady=6, sticky='w')
        tk.Label(win, text='→  신호 이름 :',
                 font=('Arial', 10)).grid(row=i+2, column=1, padx=4)
        ent = tk.Entry(win, width=22, font=('Arial', 10))
        ent.insert(0, default_name)
        ent.grid(row=i+2, column=2, padx=(0, 28), pady=6)
        entries[col] = ent

    result_holder = {}

    def on_ok():
        selected = [(col, entries[col].get().strip(), color, cmap)
                    for col, _, _, _, color, cmap in WAVE_OPTIONS
                    if checks[col].get()]
        if not selected:
            tk.Label(win, text='최소 1개 선택하세요', fg='red',
                     font=('Arial', 9)).grid(row=5, column=0, columnspan=3)
            return
        result_holder['signals'] = {
            f'ch{col[:3]}': {
                'col':   col,
                'scale': 1/100,
                'label': label or col,
                'color': color,
                'cmap':  cmap,
            }
            for col, label, color, cmap in selected
        }
        win.destroy()

    tk.Button(win, text='확인', command=on_ok,
              width=12, font=('Arial', 10)).grid(
        row=4, column=0, columnspan=3, pady=16)
    win.wait_window()
    return result_holder.get('signals', {})

SIGNAL_CONFIG = ask_signals()

# COHORT_CONFIG에 CUSTOM 항목으로 동적 등록
COHORT_CONFIG['CUSTOM'] = {
    'description': '  +  '.join(v['label'] for v in SIGNAL_CONFIG.values()),
    'signals':  SIGNAL_CONFIG,
    'primary':  list(SIGNAL_CONFIG.keys())[0],
    'figures':  ['fig3e', 'exfig2d', 'exfig2e'],
}
SELECTED_COHORT = 'CUSTOM'

print('신호 설정 완료:')
for k, v in SIGNAL_CONFIG.items():
    print(f'  {v["col"]}  ->  {v["label"]}')

## Cell 4 — 세션 로드 함수 (`load_session`)
- 경로에서 코호트 자동 감지 (`detect_cohort`)
- `roi_info.json` → `px_per_cm`, `shelter_center`, `arena_center`
- DLC 열 이름 동적 처리 (`find_body_cols`)
- `behavior/session_timestamps.csv` → 세션 `start` ~ `save` 구간만 crop (`USE_SESSION_WINDOW`)

In [ ]:
def _find_fip_files(path):
    """fip/ 안의 Raw_dFF CSV 찾기.
    구버전: Raw_dFF_Data_ROI0.csv
    신버전: <세션ID>_Raw_dFF_Data_ROI0.csv   ← 접두사가 붙는다
    """
    for pat in ('*Raw_dFF_Data_ROI*.csv', 'Raw_dFF_Data_ROI*.csv'):
        hits = sorted(glob.glob(os.path.join(path, 'fip', pat)))
        if hits:
            return hits
    return []

def detect_cohort(path):
    # Cell 3에서 GUI로 선택한 코호트 우선 사용
    if 'SELECTED_COHORT' in globals() and SELECTED_COHORT:
        return SELECTED_COHORT

    # fallback: 경로명에 코호트 키 포함 여부
    p = path.replace('\\', '/')
    for key in COHORT_CONFIG:
        if key in p: return key

    # fallback: FIP CSV 컬럼으로 자동 판단
    fip_files = _find_fip_files(path)                          # ★ 수정
    if fip_files:
        cols = pd.read_csv(fip_files[0], nrows=0).columns.tolist()
        if '565nm_dFF(%)' in cols or '565nm_dFF_corrected(%)' in cols:
            print(f'  [auto-detect] 565nm 컬럼 감지 → FIP_wt_dLight+jRGECO')
            return 'FIP_wt_dLight+jRGECO'
        if any('470nm' in c for c in cols):
            print(f'  [auto-detect] 470nm 전용 → FIP_wt_CamKII+FGCaMP')
            return 'FIP_wt_CamKII+FGCaMP'

    raise ValueError(f'알 수 없는 코호트: {path}')

def find_body_cols(dlc_raw):
    mapping = {}
    for bp, coord in dlc_raw.columns:
        if bp in ('bodyparts', 'scorer'): continue
        suffix = 'p' if coord == 'likelihood' else coord
        mapping[f'{bp}_{suffix}'] = (bp, coord)
    return mapping

def load_session(sess_path):
    cohort    = detect_cohort(sess_path)
    cfg       = COHORT_CONFIG[cohort]
    parts     = sess_path.replace('\\', '/').split('/')
    animal_id      = f"{parts[-2]} / {parts[-1]}"
    animal_id_safe = f"{parts[-2]}_{parts[-1]}"

    # ── 파일 찾기 헬퍼 ────────────────────────────────────────────────────
    #   구버전:  behavior/timeline_events.csv, 10281_topcamera_log_<ts>.csv
    #   신버전:  behavior/<세션ID>_timeline_events.csv, <세션ID>_topcamera_log.csv
    #   두 명명 규칙을 모두 매치시킨다.
    def _find(subdir, *patterns):
        for pat in patterns:
            hits = sorted(glob.glob(os.path.join(sess_path, subdir, pat)))
            if hits:
                return hits[0]
        return None

    roi_path = os.path.join(sess_path, 'roi_info.json')
    if not os.path.exists(roi_path):
        raise FileNotFoundError(f'roi_info.json 없음: {sess_path}')
    with open(roi_path, encoding='utf-8') as f:
        roi = _json.load(f)

    # ── FIP: 선택 사항 ────────────────────────────────────────────────────
    #   없으면 dff=None, fip_t=None. compute_behavior 가 카메라 시간축으로 대체하고
    #   그림은 자극(opto/threat) 기준으로 그려진다.
    fip_files = _find_fip_files(sess_path)                     # ★ 수정
    dff = pd.read_csv(fip_files[0]) if fip_files else None
    if dff is None:
        print('  FIP 없음 → 카메라 시간축 + 자극 기준으로 분석')
    else:
        print(f'  FIP 로드: {os.path.basename(fip_files[0])}')

    ev_path = _find('behavior', 'timeline_events.csv', '*timeline_events.csv')
    if ev_path is None:
        raise FileNotFoundError(f'timeline_events.csv 없음: {sess_path}')
    ev = pd.read_csv(ev_path, encoding='utf-8-sig')

    # ── 세션 시작(start) / 종료(save) 시각 ────────────────────────────────
    ts_path  = _find('behavior', 'session_timestamps.csv', '*session_timestamps.csv')
    sess_end = None
    if ts_path:
        ts = pd.read_csv(ts_path, encoding='utf-8-sig')
        ts['event']       = ts['event'].astype(str).str.strip().str.lower()
        ts['timestamp_s'] = pd.to_numeric(ts['timestamp_s'], errors='coerce')
        ts = ts.dropna(subset=['timestamp_s'])
        t_start = ts.loc[ts['event'].isin(SESSION_START_EVENTS), 'timestamp_s']
        t_end   = ts.loc[ts['event'].isin(SESSION_END_EVENTS),   'timestamp_s']
        if t_start.empty:
            raise ValueError(f'session_timestamps.csv 에 시작 이벤트 없음: {ts_path}')
        sess_start = float(t_start.iloc[0])
        sess_end   = float(t_end.iloc[-1]) if not t_end.empty else None
        if sess_end is None:
            print('  경고: 세션 종료 이벤트 없음 — 전체 구간 사용')
        elif sess_end <= sess_start:
            print(f'  경고: 세션 종료({sess_end:.1f}) <= 시작({sess_start:.1f}) — crop 생략')
            sess_end = None
    else:
        json_files = [f for f in sorted(glob.glob(os.path.join(sess_path, 'behavior', '*.json')))
                      if 'session_window' not in os.path.basename(f)]
        if not json_files:
            raise FileNotFoundError(f'session_timestamps.csv / JSON 없음: {sess_path}')
        with open(json_files[0], encoding='utf-8') as f:
            meta = _json.load(f)
        from datetime import datetime
        dt = datetime.fromisoformat(meta['Other_SessionStartTime'])
        sess_start = dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1e6
        print('  경고: session_timestamps.csv 없음 — JSON 시작시각 사용, 전체 구간 분석')

    if 'elapsed_ms' in ev.columns and 'start_timestamp_s' not in ev.columns:
        ev['start_timestamp_s'] = sess_start + ev['elapsed_ms'] / 1000.0
        ev['end_timestamp_s']   = sess_start + ev['end_ms']     / 1000.0

    # ── 카메라 로그 ───────────────────────────────────────────────────────
    cam_path = _find('behavior', '*_log_*.csv', '*_log.csv', '*camera*log*.csv')
    if cam_path is None:
        raise FileNotFoundError(f'카메라 로그 없음: {sess_path}')
    cam = pd.read_csv(cam_path, header=None,
                      names=['frame_id', 'type', 'timestamp_s'])
    # 헤더 행이 섞여 들어와도(= DLC csv 오선택) 조용히 죽지 않도록 수치화 후 정리
    cam['timestamp_s'] = pd.to_numeric(cam['timestamp_s'], errors='coerce')
    cam = cam.dropna(subset=['timestamp_s']).reset_index(drop=True)
    if len(cam) < 10:
        raise ValueError(f'카메라 로그에 유효한 timestamp 가 없음: {cam_path}')
    cam['rel_s'] = cam['timestamp_s'].values - sess_start

    dlc_files = glob.glob(os.path.join(sess_path, 'behavior-videos', '*_filtered.csv'))
    if not dlc_files:
        dlc_files = [f for f in glob.glob(os.path.join(sess_path, 'behavior-videos', '*DLC*.csv'))
                     if '_log_' not in f]
    if not dlc_files: raise FileNotFoundError(f'DLC 없음: {sess_path}')
    dlc_raw = pd.read_csv(dlc_files[0], header=[1, 2])
    dlc     = pd.DataFrame({k: dlc_raw[v] for k, v in find_body_cols(dlc_raw).items()})

    if dff is not None:
        time_col = 'Computer Time (s)' if 'Computer Time (s)' in dff.columns else 'Time (s)'
        fip_t    = dff[time_col].values.astype(float)
        if time_col == 'Computer Time (s)':
            fip_t = fip_t - sess_start
    else:
        fip_t = None

    # ── 세션 구간 [start, save] 으로 crop ─────────────────────────────────
    #   시간축은 세션 시작이 0 이므로 0 ~ sess_dur 만 남긴다.
    #   cam ↔ dlc 는 행 위치로 대응하므로 길이를 맞춘 뒤 같은 mask 를 적용.
    sess_dur = None
    if USE_SESSION_WINDOW and sess_end is not None:
        sess_dur = sess_end - sess_start

        n_fr = min(len(cam), len(dlc))
        cam  = cam.iloc[:n_fr].reset_index(drop=True)
        dlc  = dlc.iloc[:n_fr].reset_index(drop=True)
        m_fr = (cam['rel_s'].values >= 0) & (cam['rel_s'].values <= sess_dur)
        if m_fr.sum() > 10:
            cam = cam[m_fr].reset_index(drop=True)
            dlc = dlc[m_fr].reset_index(drop=True)
        else:
            print('  경고: 카메라 프레임이 세션 구간과 겹치지 않음 — 프레임 crop 생략')

        if fip_t is not None:
            m_fip = (fip_t >= 0) & (fip_t <= sess_dur)
            if m_fip.sum() > 10:
                dff   = dff[m_fip].reset_index(drop=True)
                fip_t = fip_t[m_fip]
            else:
                print('  경고: FIP 샘플이 세션 구간과 겹치지 않음 — FIP crop 생략')

        print(f'  세션 구간 0 – {sess_dur:.1f} s  '
              f'(cam {len(cam):,} fr, DLC {len(dlc):,} fr, '
              + (f'FIP {len(fip_t):,} sa)' if fip_t is not None else 'FIP 없음)'))

    fps_fip = 1.0 / float(np.median(np.diff(fip_t))) if fip_t is not None else None

    return dict(path=sess_path, animal_id=animal_id, animal_id_safe=animal_id_safe,
                cohort=cohort, cfg=cfg, roi=roi, dff=dff, ev=ev, cam=cam, dlc=dlc,
                sess_start=sess_start, sess_end=sess_end, sess_dur=sess_dur,
                fip_t=fip_t, fps_fip=fps_fip)

print('load_session() OK')


## Cell 5 — 신호 처리 함수
| 함수 | 역할 |
|------|------|
| `compute_geometry` | roi_info.json 기반 좌표계 + shelter polygon |
| `find_bouts` | 최소 지속시간 조건 충족 구간 추출 |
| `compute_behavior` | 거리·속도·Rest/Move/Shelter 산출 |
| `normalize_signals` | z-score + 200 ms smoothing |
| `get_event_times` | 위협·shelter제거·opto 어노테이션 추출 |
| `extract_trials` | peri-event 행렬 생성 |

In [ ]:
# ── compute_geometry : roi_info.json 기반 ────────────────────────────────
def compute_geometry(sess):
    roi = sess['roi']
    dlc = sess['dlc']
    for candidate in ('body_center_x', 'body_x'):
        if candidate in dlc.columns:
            bx_col = candidate
            break
    else:
        bx_col = next(c for c in dlc.columns if c.endswith('_x'))
    print(f'  position bodypart: {bx_col.replace("_x", "")}')
    shelter_poly_px = (np.array(roi['shelter_points'], dtype=float)
                       if 'shelter_points' in roi else None)
    return dict(
        px_per_cm       = roi['px_per_cm'],
        arena_cx        = roi['arena_center'][0],
        arena_cy        = roi['arena_center'][1],
        shelter_cx      = roi['shelter_center'][0],
        shelter_cy      = roi['shelter_center'][1],
        shelter_poly_px = shelter_poly_px,
        bx_col = bx_col,
        by_col = bx_col.replace('_x', '_y'),
        bp_col = bx_col.replace('_x', '_p'),
    )


# ── find_bouts ────────────────────────────────────────────────────────────
def find_bouts(mask, t, min_dur_s):
    if not np.any(mask):
        return np.array([]), np.array([])
    dt    = float(np.median(np.diff(t)))
    min_n = max(1, int(np.round(min_dur_s / dt)))
    pad   = np.r_[False, mask, False].astype(int)
    d     = np.diff(pad)
    ons   = np.where(d ==  1)[0]
    offs  = np.where(d == -1)[0]
    keep  = (offs - ons) >= min_n
    if not np.any(keep):
        return np.array([]), np.array([])
    return t[ons[keep]], t[np.clip(offs[keep] - 1, 0, len(t) - 1)]


# ── compute_behavior ──────────────────────────────────────────────────────
DLC_LIKELI_THR  = 0.5    # likelihood 미만 프레임 → 보간
SPD_JUMP_THR    = 500.0  # cm/s 초과 프레임 → 비정상 tracking jump → 보간
DIST_SMOOTH_N   = 3      # 거리용 위치 스무딩 (프레임)
SPEED_WIN       = 5      # 속도 계산 displacement 윈도우 (프레임)

def compute_behavior(sess, geom):
    dlc   = sess['dlc']; cam = sess['cam']
    fip_t = sess['fip_t']          # FIP 있으면 FIP 시간, 없으면 None
    ppc   = geom['px_per_cm']
    scx   = geom['shelter_cx']; scy = geom['shelter_cy']
    shelter_poly_px = geom.get('shelter_poly_px')
    bx    = dlc[geom['bx_col']].values.astype(float)
    by    = dlc[geom['by_col']].values.astype(float)
    bp_col = geom['bp_col']
    bp    = dlc[bp_col].values.astype(float) if bp_col in dlc.columns else np.ones(len(bx))
    cam_t = cam['rel_s'].values[:len(bx)]
    n     = min(len(cam_t), len(bx))
    cam_t = cam_t[:n]; bx = bx[:n]; by = by[:n]; bp = bp[:n]

    dt_c = float(np.median(np.diff(cam_t)))
    xi   = np.arange(n)

    # ── 1차 필터: likelihood 낮은 프레임 보간 ────────────────────────────
    good = (bp >= DLC_LIKELI_THR) & np.isfinite(bx) & np.isfinite(by)
    if good.sum() > 2:
        bx = np.interp(xi, xi[good], bx[good])
        by = np.interp(xi, xi[good], by[good])

    # ── 2차 필터: speed 500 cm/s 초과 프레임 보간 (최대 3회 반복) ────────
    n_removed = 0
    for _ in range(3):
        dx1  = np.r_[0, np.diff(bx)] / ppc
        dy1  = np.r_[0, np.diff(by)] / ppc
        spd1 = np.sqrt(dx1**2 + dy1**2) / dt_c
        bad  = spd1 > SPD_JUMP_THR
        if not bad.any():
            break
        n_removed += int(bad.sum())
        good2 = ~bad & np.isfinite(bx)
        if good2.sum() > 2:
            bx = np.interp(xi, xi[good2], bx[good2])
            by = np.interp(xi, xi[good2], by[good2])

    # ── 거리: 스무딩 위치 사용 ───────────────────────────────────────────
    bx_sm   = uniform_filter1d(bx, size=DIST_SMOOTH_N)
    by_sm   = uniform_filter1d(by, size=DIST_SMOOTH_N)
    dist_cm = np.sqrt((bx_sm - scx)**2 + (by_sm - scy)**2) / ppc

    # ── 속도: N프레임 간격 displacement ──────────────────────────────────
    dx  = (bx[SPEED_WIN:] - bx[:-SPEED_WIN]) / ppc
    dy  = (by[SPEED_WIN:] - by[:-SPEED_WIN]) / ppc
    spd_raw = np.sqrt(dx**2 + dy**2) / (SPEED_WIN * dt_c)
    half = SPEED_WIN // 2
    spd  = np.r_[np.full(half, spd_raw[0]),
                 spd_raw,
                 np.full(SPEED_WIN - half, spd_raw[-1])]

    valid = np.isfinite(cam_t) & np.isfinite(dist_cm) & np.isfinite(spd)
    ct = cam_t[valid]

    # FIP 없으면 카메라 시간축을 공통 시간축으로 사용
    if fip_t is None:
        fip_t = ct
        sess['fip_t']   = fip_t
        sess['fps_fip'] = 1.0 / float(np.median(np.diff(ct)))
        print(f'  카메라 시간축 사용: {len(fip_t):,} frames @ {sess["fps_fip"]:.1f} Hz')

    dist_fip = interp1d(ct, dist_cm[valid], bounds_error=False,
                        fill_value=(dist_cm[valid][0], dist_cm[valid][-1]))(fip_t)
    spd_fip  = interp1d(ct, spd[valid],     bounds_error=False,
                        fill_value=(spd[valid][0], spd[valid][-1]))(fip_t)

    # ── behavior 검출 ─────────────────────────────────────────────────────
    is_rest   = spd_fip < BEHAV['rest_thr']
    is_moving = spd_fip > BEHAV['move_thr']

    # Shelter: polygon 기반 (없으면 거리 기반 fallback)
    if shelter_poly_px is not None:
        pts_px    = np.column_stack([bx, by])
        in_sh_raw = MplPath(shelter_poly_px).contains_points(pts_px).astype(float)
        in_shelter_fip = interp1d(ct, in_sh_raw[valid], bounds_error=False,
                                  fill_value=0.0)(fip_t) > 0.5
        shelter_on, shelter_off = find_bouts(in_shelter_fip, fip_t, BEHAV['shelter_dur'])
    else:
        shelter_on, shelter_off = find_bouts(
            dist_fip < BEHAV['shelter_r'], fip_t, BEHAV['shelter_dur'])

    # ── Genuine shelter entry ─────────────────────────────────────────────
    dt_fip   = float(np.median(np.diff(fip_t)))
    lb_n     = max(1, int(np.round(BEHAV['genuine_entry_lookback'] / dt_fip)))
    min_dist = BEHAV['genuine_entry_min_dist']

    genuine_mask = []
    for t_on in shelter_on:
        idx   = int(np.searchsorted(fip_t, t_on))
        start = max(0, idx - lb_n)
        prev  = dist_fip[start:idx]
        genuine_mask.append(len(prev) > 0 and float(prev.max()) > min_dist)

    shelter_genuine_on  = shelter_on[np.array(genuine_mask, dtype=bool)]
    shelter_genuine_off = shelter_off[np.array(genuine_mask, dtype=bool)]

    print(f'    jump 제거: {n_removed} frames  |  shelter stay: {len(shelter_on)}  genuine entry: {len(shelter_genuine_on)}')

    return dict(
        dist_fip            = dist_fip,
        spd_fip             = spd_fip,
        is_rest             = is_rest,
        is_moving           = is_moving,
        shelter_on          = shelter_on,
        shelter_off         = shelter_off,
        shelter_genuine_on  = shelter_genuine_on,
        shelter_genuine_off = shelter_genuine_off,
    )


# ── normalize_signals ─────────────────────────────────────────────────────
def normalize_signals(sess):
    dff = sess['dff']; cfg = sess['cfg']
    if dff is None:
        print('  FIP 없음 — 신호 정규화 스킵 (행동 지표만 사용)')
        return {}

    result = {}
    for sig_key, sc in cfg['signals'].items():
        col = sc['col']
        if col not in dff.columns:
            print(f'  경고: {col} 열 없음 — 스킵')
            continue
        raw = dff[col].values.astype(float) * sc['scale']
        result[sig_key] = (raw - np.nanmean(raw)) / np.nanstd(raw)
        print(f'  {sig_key} ({col}): z-score 적용')

    return result


# ── get_event_times ───────────────────────────────────────────────────────
def get_event_times(sess):
    """세션 시작 기준 상대시각(s). sess_dur 이 있으면 세션 구간 밖 이벤트는 제외."""
    ev  = sess['ev']; ss = sess['sess_start']
    dur = sess.get('sess_dur')

    def find(pat):
        sel = ev[ev['event'].astype(str).str.contains(pat, case=False, na=False)]
        on  = pd.to_numeric(sel['start_timestamp_s'], errors='coerce').values - ss
        if 'end_timestamp_s' in sel.columns:
            off = pd.to_numeric(sel['end_timestamp_s'], errors='coerce').values - ss
        else:
            off = np.full(len(on), np.nan)
        keep = np.isfinite(on)
        if dur is not None:
            keep = keep & (on >= 0) & (on <= dur)
        on, off = on[keep], off[keep]
        off = np.where(np.isfinite(off), off, on)   # point 이벤트는 off = on
        if dur is not None:
            off = np.clip(off, 0.0, dur)
        return on, off

    threat_on, threat_off = find(r'Threat')
    shelter_rm, _         = find(r'Remove|remov')
    opto_on,   opto_off   = find(r'Opto')
    evts = dict(
        threat_on  = threat_on,
        threat_off = threat_off,
        shelter_rm = shelter_rm,
        opto_on    = opto_on,
        opto_off   = opto_off,
    )
    if len(evts['opto_on']):
        print(f'  Opto events: {len(evts["opto_on"])}개  '
              f'({evts["opto_on"][0]:.1f} ~ {evts["opto_on"][-1]:.1f} s)')
    return evts


# ── extract_trials ────────────────────────────────────────────────────────
def extract_trials(sig, t, events_s, pre_s, post_s, fs):
    pre_n  = int(np.round(pre_s  * fs))
    post_n = int(np.round(post_s * fs))
    n      = pre_n + post_n
    trials = []
    for et in events_s:
        idx = int(np.searchsorted(t, et))
        s, e = idx - pre_n, idx + post_n
        if 0 <= s and e <= len(sig):
            trials.append(sig[s:e])
    t_ax = np.linspace(-pre_s, post_s, n, endpoint=False)
    if not trials: return np.zeros((0, n)), t_ax
    return np.vstack(trials), t_ax


print('모든 함수 정의 완료')
print(f'  SPD_JUMP_THR={SPD_JUMP_THR} cm/s | DIST_SMOOTH_N={DIST_SMOOTH_N} | SPEED_WIN={SPEED_WIN}')

## Cell 6 — 전체 세션 처리 루프
선택된 세션을 순서대로 처리해 `session_data` 리스트에 누적합니다. 오류 발생 시 해당 세션만 건너뜁니다.

In [ ]:
session_data = []

for sess_path in selected_sessions:
    print(f'\n처리 중: {sess_path}')
    try:
        sess = load_session(sess_path)
        geom = compute_geometry(sess)
        beh  = compute_behavior(sess, geom)
        sigs = normalize_signals(sess)
        evts = get_event_times(sess)
        sess.update(geom=geom, beh=beh, sigs=sigs, evts=evts)
        session_data.append(sess)

        r = sess['roi']
        print(f'  OK  {sess["animal_id"]} ({sess["cohort"]})')
        print(f'      px_per_cm={r["px_per_cm"]:.3f}  '
              f'shelter_center=({r["shelter_center"][0]:.0f},{r["shelter_center"][1]:.0f})px  '
              f'shelter_poly={"있음" if geom.get("shelter_poly_px") is not None else "없음(거리 기반)"}')
        if sess['dff'] is not None:
            print(f'      FIP {len(sess["fip_t"]):,} sa @ {sess["fps_fip"]:.1f} Hz  '
                  f'signals={list(sigs.keys())}')
        else:
            print(f'      FIP 없음 — 카메라 시간축 {len(sess["fip_t"]):,} fr '
                  f'@ {sess["fps_fip"]:.1f} Hz')
        _dur = sess.get('sess_dur')
        print('      세션 구간: '
              + (f'0 – {_dur:.1f} s' if _dur else '전체 (session_timestamps 없음)'))
        print(f'      Shelter stays={len(beh["shelter_on"])}  '
              f'Threats={len(evts["threat_on"])}')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  ERROR: {e}')

print(f'\n총 {len(session_data)}개 세션 완료')

## Cell 7 — SessionOverview
3-패널 구성: **신호** (shelter=파랑·threat=빨강·opto=초록) / **shelter까지 거리** / **속도** (2·3 cm/s 기준선)

In [ ]:
# SessionOverview — 전체 세션 개요
for sess in session_data:
    cfg = sess['cfg']
    if 'fig3e' not in cfg['figures']: continue

    fip_t = sess['fip_t']; beh = sess['beh']; evts = sess['evts']
    sig_keys = [k for k in cfg['signals'] if k in sess['sigs']]
    n_sigs = len(sig_keys)

    SHELTER_C = '#88aaee'; THREAT_C = '#ff5555'
    REMOVE_C  = '#aa00aa'; OPTO_C   = '#00bb77'

    if n_sigs > 0:
        height_ratios = [2] * n_sigs + [1.2, 1.2]
        n_rows = n_sigs + 2
    else:
        height_ratios = [1.5, 1.5]
        n_rows = 2

    fig, axes = plt.subplots(n_rows, 1,
                              figsize=(14, 2 + 2 * max(n_sigs, 1)),
                              sharex=True,
                              gridspec_kw={'height_ratios': height_ratios, 'hspace': 0.08})
    axes = np.atleast_1d(axes)

    sess_id = sess['animal_id'].split('/')[-1].strip()
    _title = f"{sess_id}  ({cfg['description']})" if n_sigs else f"{sess_id}  (behavior only)"
    fig.suptitle(_title, fontsize=14, y=1.01)

    for ax in axes:
        for on, off in zip(beh['shelter_on'], beh['shelter_off']):
            ax.axvspan(on, off, color=SHELTER_C, alpha=0.30, lw=0)
        for on, off in zip(evts['threat_on'], evts['threat_off']):
            ax.axvspan(on, off, color=THREAT_C, alpha=0.35, lw=0)
        for t in evts['shelter_rm']:
            ax.axvline(t, color=REMOVE_C, lw=1.0, ls='--')
        for on, off in zip(evts['opto_on'], evts['opto_off']):
            ax.axvspan(on, off, color=OPTO_C, alpha=0.45, lw=0)
        ax.margins(x=0)
        ax.tick_params(axis='y', labelsize=14)

    # 신호 패널
    for si, sig_key in enumerate(sig_keys):
        sc = cfg['signals'][sig_key]
        z  = sess['sigs'][sig_key]
        axes[si].plot(fip_t, z, color=sc['color'], lw=0.5, rasterized=True)
        axes[si].axhline(0, color='#aaa', lw=0.4, ls=':')
        axes[si].set_ylabel(f"{sc['label']}\n(z-score)", fontsize=14, labelpad=8)

    # 범례 — 그래프 밖 우측 상단
    legend_handles = [
        mpatches.Patch(color=SHELTER_C, alpha=0.6, label='Shelter'),
        mpatches.Patch(color=THREAT_C,  alpha=0.6, label='Threat'),
        plt.Line2D([0],[0], color=REMOVE_C, ls='--', lw=2, label='Shelter removed'),
    ]
    if len(evts['opto_on']):
        legend_handles.append(
            mpatches.Patch(color=OPTO_C, alpha=0.7, label='Optogenetics'))

    axes[0].legend(handles=legend_handles, fontsize=14, frameon=False,
                   bbox_to_anchor=(1.01, 1.0), loc='upper left', ncol=1,
                   handlelength=2.0, handleheight=1.0, handletextpad=0.8)

    # 거리 패널
    ax_dist = axes[n_sigs]
    ax_dist.plot(fip_t, beh['dist_fip'], color='#333', lw=0.5, rasterized=True)
    ax_dist.set_ylabel('Dist.\n(cm)', fontsize=14, labelpad=8)
    ax_dist.set_ylim(bottom=0)

    # 속도 패널
    ax_spd = axes[n_sigs + 1]
    ax_spd.plot(fip_t, beh['spd_fip'], color='#555', lw=0.5, rasterized=True)
    ax_spd.set_ylabel('Speed\n(cm/s)', fontsize=14, labelpad=8)
    ax_spd.set_ylim(bottom=0)
    ax_spd.set_xlabel('Time (s)', fontsize=14)
    ax_spd.tick_params(axis='x', labelsize=14)

    fig.align_ylabels(axes)
    plt.tight_layout()

    # x축을 세션 시작(0) ~ 세션 종료(sess_dur) 로 고정
    # set_xticks 가 view limit 을 넓히므로 반드시 xlim 을 나중에 적용한다.
    t_end = sess.get('sess_dur') or float(fip_t[-1])
    ax_spd.set_xticks([t for t in ax_spd.get_xticks() if 0 < t <= t_end])
    for ax in axes:
        ax.set_xlim(0, t_end)

    fname = os.path.join(SAVE_DIR, f"{sess['animal_id_safe']}_fig3e")
    plt.savefig(fname+'.pdf', bbox_inches='tight')
    plt.savefig(fname+'.png', dpi=200, bbox_inches='tight')
    plt.show(); print('저장:', fname)

In [ ]:
# 모든 opto 자극 전후 20초 — 한 figure에 자극별 열 배치 (위: dist, 아래: speed)
PRE, POST = 20.0, 20.0
OPTO_C = '#00bb77'

for sess in session_data:
    evts     = sess['evts']
    opto_on  = evts['opto_on']
    opto_off = evts['opto_off']
    if len(opto_on) == 0:
        print(f"{sess['animal_id']}: opto 이벤트 없음"); continue

    n_opto = len(opto_on)
    fip_t  = sess['fip_t']
    beh    = sess['beh']

    fig, axes = plt.subplots(2, n_opto,
                              figsize=(3.5 * n_opto, 5),
                              sharex='col', sharey='row',
                              gridspec_kw={'hspace': 0.08, 'wspace': 0.25})
    axes = np.atleast_2d(axes)
    fig.suptitle(f"All Opto  {sess['animal_id']}  (±{int(PRE)} s each)",
                 fontsize=9, y=1.01)

    for ci, (t0, t1) in enumerate(zip(opto_on, opto_off)):
        mask  = (fip_t >= t0 - PRE) & (fip_t <= t0 + POST)
        t_rel = fip_t[mask] - t0
        dur   = t1 - t0

        ax_d = axes[0, ci]
        ax_s = axes[1, ci]

        for ax in (ax_d, ax_s):
            ax.axvspan(0, dur, color=OPTO_C, alpha=0.40, lw=0)
            ax.axvline(0, color=OPTO_C, lw=1.0, ls='-')

        ax_d.plot(t_rel, beh['dist_fip'][mask], color='#333', lw=0.7)
        ax_d.axhline(BEHAV['shelter_r'], color='#88aaee', lw=0.7, ls='--')
        ax_d.set_title(f"#{ci+1}  t={t0:.0f}s", fontsize=7)
        if ci == 0:
            ax_d.set_ylabel('Dist. to shelter (cm)', fontsize=8)
        ax_d.set_ylim(bottom=0)

        ax_s.plot(t_rel, beh['spd_fip'][mask], color='#555', lw=0.7)
        for thr, col in [
            (BEHAV['rest_thr'],  'steelblue'),
            (BEHAV['move_thr'],  '#e67c00'),
        ]:
            ax_s.axhline(thr, color=col, lw=0.6, ls='--')
        if ci == 0:
            ax_s.set_ylabel('Speed (cm/s)', fontsize=8)
        ax_s.set_ylim(bottom=0)
        ax_s.set_xlabel('Time (s)', fontsize=7)

    plt.tight_layout()
    fname = os.path.join(SAVE_DIR, f"{sess['animal_id_safe']}_opto_all")
    plt.savefig(fname + '.pdf', bbox_inches='tight')
    plt.savefig(fname + '.png', dpi=200, bbox_inches='tight')
    plt.show(); print('저장:', fname)

#Fig 3e — 1분 구간 줌인 × 6개

In [ ]:
# SessionOverview — 40s 구간 줌인 × 6개
WIN = 20.0   # ±20 s → 총 40s

for sess in session_data:
    cfg = sess['cfg']
    if 'fig3e' not in cfg['figures']: continue

    fip_t  = sess['fip_t']; beh = sess['beh']; evts = sess['evts']
    sig_keys = [k for k in cfg['signals'] if k in sess['sigs']]
    n_sigs   = len(sig_keys)

    # threat 이 없는 세션(opto 전용)은 자극(opto) 기준으로 줌인한다.
    if len(evts['threat_on']) > 0:
        centers_all, ev_name = evts['threat_on'], 'Threat'
    elif len(evts['opto_on']) > 0:
        centers_all, ev_name = evts['opto_on'],   'Opto'
    else:
        print(f"{sess['animal_id']}: threat / opto 이벤트 없음"); continue

    centers = sorted(centers_all)[:6]

    SHELTER_C = '#88aaee'; THREAT_C = '#ff5555'
    REMOVE_C  = '#aa00aa'; OPTO_C   = '#00bb77'
    height_ratios = [2] * n_sigs + [1.2, 1.2]

    sess_id  = sess['animal_id'].split('/')[-1].strip()
    sig_desc = '  +  '.join(cfg['signals'][k]['label'] for k in sig_keys)

    for fi, t_center in enumerate(centers):
        t_start = t_center - WIN
        t_end   = t_center + WIN
        mask = (fip_t >= t_start) & (fip_t <= t_end)
        if not np.any(mask): continue

        fig, axes = plt.subplots(n_sigs + 2, 1,
                                  figsize=(14, 2 + 2 * max(n_sigs, 1)),
                                  sharex=True,
                                  gridspec_kw={'height_ratios': height_ratios, 'hspace': 0.08})
        fig.suptitle(
            (f"{sess_id}  [{sig_desc}]" if sig_desc else f"{sess_id}")
            + f"  [{ev_name} {fi+1}/{len(centers)}]  {t_start:.0f}–{t_end:.0f} s",
            fontsize=14, y=1.01)

        for ax in axes:
            for on, off in zip(beh['shelter_on'], beh['shelter_off']):
                if off >= t_start and on <= t_end:
                    ax.axvspan(max(on, t_start), min(off, t_end),
                               color=SHELTER_C, alpha=0.30, lw=0)
            for on, off in zip(evts['threat_on'], evts['threat_off']):
                if off >= t_start and on <= t_end:
                    ax.axvspan(max(on, t_start), min(off, t_end),
                               color=THREAT_C, alpha=0.35, lw=0)
            for on, off in zip(evts['opto_on'], evts['opto_off']):
                if off >= t_start and on <= t_end:
                    ax.axvspan(max(on, t_start), min(off, t_end),
                               color=OPTO_C, alpha=0.45, lw=0)
            for t in evts['shelter_rm']:
                if t_start <= t <= t_end:
                    ax.axvline(t, color=REMOVE_C, lw=1.0, ls='--')
            ax.set_xlim(t_start, t_end)
            ax.tick_params(axis='y', labelsize=14)

        for si, sig_key in enumerate(sig_keys):
            sc = cfg['signals'][sig_key]
            axes[si].plot(fip_t[mask], sess['sigs'][sig_key][mask],
                          color=sc['color'], lw=0.7)
            axes[si].axhline(0, color='#aaa', lw=0.4, ls=':')
            axes[si].set_ylabel(f"{sc['label']}\n(z-score)", fontsize=14, labelpad=8)

        # 범례 — 첫 패널 상단 오른쪽 (그래프 안쪽 끝에 붙임)
        _handles = [
            mpatches.Patch(color=SHELTER_C, alpha=0.6, label='Shelter'),
        ]
        if len(evts['threat_on']):
            _handles.append(mpatches.Patch(color=THREAT_C, alpha=0.6, label='Threat'))
        if len(evts['opto_on']):
            _handles.append(mpatches.Patch(color=OPTO_C, alpha=0.7, label='Optogenetics'))
        _handles.append(
            plt.Line2D([0], [0], color=REMOVE_C, ls='--', lw=1, label='Shelter removed'))
        axes[0].legend(handles=_handles, fontsize=10, frameon=False,
           bbox_to_anchor=(1.0, 1.0), loc='lower right', ncol=3,
           handlelength=2.0, handletextpad=0.8)

        ax_dist = axes[n_sigs]
        ax_dist.plot(fip_t[mask], beh['dist_fip'][mask], color='#333', lw=0.7)
        ax_dist.set_ylabel('Dist.\n(cm)', fontsize=14, labelpad=8)
        ax_dist.set_ylim(bottom=0)

        ax_spd = axes[n_sigs + 1]
        ax_spd.plot(fip_t[mask], beh['spd_fip'][mask], color='#555', lw=0.7)
        ax_spd.set_ylabel('Speed\n(cm/s)', fontsize=14, labelpad=8)
        ax_spd.set_ylim(bottom=0)
        ax_spd.set_xlabel('Time (s)', fontsize=14)
        ax_spd.tick_params(axis='x', labelsize=14)

        fig.align_ylabels(axes)
        plt.tight_layout()
        fname = os.path.join(SAVE_DIR,
                             f"{sess['animal_id_safe']}_fig3e_zoom{fi+1:02d}"
                             f"_{t_start:.0f}-{t_end:.0f}s")
        plt.savefig(fname + '.pdf', bbox_inches='tight')
        plt.savefig(fname + '.png', dpi=200, bbox_inches='tight')
        plt.show()
        print(f'저장: {fname}')

## Cell 8 — Shelter Entry 정렬 히트맵
**공간 기반** shelter 입장 기준 (1 s 이상). 각 trial을 peak latency 순으로 정렬 + Mean±SEM trace.

In [ ]:
# ShelterEntry_Heatmap — shelter 입장 정렬 히트맵 (전체 entry)
for sess in session_data:
    cfg = sess['cfg']
    if 'exfig2d' not in cfg['figures']: continue

    fip_t   = sess['fip_t']; beh = sess['beh']
    entries = beh['shelter_on']
    if len(entries) == 0:
        print(f"{sess['animal_id']}: shelter entry 없음"); continue

    PRE, POST    = 5.0, 10.0
    BASELINE_WIN = (-5.0, -1.0)
    fs = sess['fps_fip']

    plot_sigs = dict(sess['sigs'])
    if not plot_sigs:
        print(f"{sess['animal_id']}: FIP 신호 없음 — shelter entry 히트맵 스킵 "
              f"(거리/속도 정렬은 ThreatAligned_Heatmap 셀에서 생성됨)")
        continue
    sig_meta  = dict(cfg['signals'])
    if 'iso' in plot_sigs:
        sig_meta['iso'] = {
            'label': '415nm isosbestic\n(self-corrected: Flat = validated)',
            'color': '#888888',
        }

    sess_id  = sess['animal_id'].split('/')[-1].strip()
    n_panels = len(plot_sigs)
    fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 5))
    axes = np.atleast_1d(axes)
    fig.suptitle(
        f"{sess_id}  Shelter Entry  n={len(entries)}",
        fontsize=13)

    for si, (sig_key, z_sig) in enumerate(plot_sigs.items()):
        sc = sig_meta[sig_key]

        mat, t_ax = extract_trials(z_sig, fip_t, entries, PRE, POST, fs)
        if mat.shape[0] == 0: continue

        bl_mask = (t_ax >= BASELINE_WIN[0]) & (t_ax <= BASELINE_WIN[1])
        bl_mean = mat[:, bl_mask].mean(axis=1, keepdims=True)
        mat_bl  = mat - bl_mean

        if sig_key == 'iso':
            mat_sorted = mat_bl
            vmax = 1.0
        else:
            mat_sorted = mat_bl          # 시간순(shelter entry 발생 순서) 유지
            vmax = float(np.nanpercentile(np.abs(mat_bl), 98))

        vmin = -vmax * 0.3
        vmid = (vmin + vmax) / 2

        ax_h = axes[si]
        im = ax_h.imshow(
            mat_sorted, aspect='auto', cmap='viridis',
            interpolation='none',
            vmin=vmin, vmax=vmax,
            extent=[t_ax[0], t_ax[-1], mat_sorted.shape[0] + .5, .5],
            origin='upper')
        ax_h.axvline(0, color='white', lw=1, ls='--')
        ax_h.set_xlabel('Time from shelter entry (s)', fontsize=14)
        ax_h.set_ylabel('Trial', fontsize=14)
        ax_h.set_title(sc['label'], fontsize=13)
        ax_h.tick_params(labelsize=12)

        cb = plt.colorbar(im, ax=ax_h, fraction=0.06, pad=0.02, shrink=0.5)
        cb.set_ticks([vmin, vmid, vmax])
        cb.set_ticklabels([f'{vmin:.2f}', f'{vmid:.2f}', f'{vmax:.2f}'])
        cb.ax.tick_params(labelsize=11)
        cb.set_label('')

    plt.tight_layout()
    os.makedirs(SAVE_DIR, exist_ok=True)
    fname = os.path.join(SAVE_DIR, f"{sess['animal_id_safe']}_shelterentry_heatmap")
    plt.savefig(fname + '.pdf', bbox_inches='tight')
    plt.savefig(fname + '.png', dpi=200, bbox_inches='tight')
    plt.show(); print('저장:', fname)

## Cell 9 — ThreatAligned_Heatmap
위협 onset 기준 정렬 (없으면 opto onset). 거리·속도·신호 히트맵 + Mean±SEM.

In [ ]:
# ThreatAligned_Heatmap — 이벤트 정렬 히트맵 (거리 + 속도 + FIP)
for sess in session_data:
    fip_t = sess['fip_t']; beh = sess['beh']; evts = sess['evts']
    cfg   = sess['cfg'];   fs  = sess['fps_fip']

    if len(evts.get('threat_on', [])) > 0:
        events   = evts['threat_on']
        ev_label = 'Threat Onset'
        ev_color = '#ff5555'
    elif len(evts.get('opto_on', [])) > 0:
        events   = evts['opto_on']
        ev_label = 'Opto Onset'
        ev_color = '#00bb77'
    else:
        print(f"{sess['animal_id']}: 이벤트 없음 (threat / opto)"); continue

    PRE, POST = 5.0, 15.0
    dist_mat, t_ax = extract_trials(beh['dist_fip'], fip_t, events, PRE, POST, fs)
    spd_mat,  _    = extract_trials(beh['spd_fip'],  fip_t, events, PRE, POST, fs)

    sig_keys = [k for k in cfg['signals'] if k in sess.get('sigs', {})]
    n_cols   = 2 + len(sig_keys)
    sess_id  = sess['animal_id'].split('/')[-1].strip()

    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 5),
                              gridspec_kw={'hspace': 0.5, 'wspace': 0.4})
    axes = np.atleast_2d(axes)
    fig.suptitle(f"{sess_id}  {ev_label}  n={len(events)}",
                 fontsize=13, y=1.01)
    ext = [t_ax[0], t_ax[-1], len(events) + .5, .5]

    # 거리 / 속도 열 (col 0, 1)
    for ci, (mat, title, color, unit) in enumerate([
        (dist_mat, 'Dist. to shelter', '#333333', 'cm'),
        (spd_mat,  'Speed',            '#e67c00', 'cm/s'),
    ]):
        vmax = float(np.nanpercentile(mat, 99))
        im = axes[0, ci].imshow(mat, aspect='auto', cmap='viridis',
                                interpolation='none',
                                vmin=0, vmax=vmax, extent=ext, origin='upper')
        axes[0, ci].axvline(0, color='white', lw=1, ls='--')
        axes[0, ci].set_title(title, fontsize=13)
        axes[0, ci].set_xlabel(f'Time from {ev_label.lower()} (s)', fontsize=14)
        axes[0, ci].set_ylabel('Trial', fontsize=14)
        axes[0, ci].tick_params(labelsize=12)
        cb = plt.colorbar(im, ax=axes[0, ci], label=unit, fraction=0.05, pad=0.02)
        cb.set_label(unit, fontsize=12)
        cb.ax.tick_params(labelsize=11)

        m = mat.mean(0); s = mat.std(0) / np.sqrt(len(mat))
        axes[1, ci].plot(t_ax, m, color=color, lw=1.5)
        axes[1, ci].fill_between(t_ax, m - s, m + s, color=color, alpha=0.25)
        axes[1, ci].axvline(0, color=ev_color, lw=0.9, ls='--')
        axes[1, ci].set_xlabel(f'Time from {ev_label.lower()} (s)', fontsize=14)
        axes[1, ci].set_ylabel(unit, fontsize=14)
        axes[1, ci].tick_params(labelsize=12)

    # FIP 신호 열 (col 2+)
    for ci, sig_key in enumerate(sig_keys):
        col = 2 + ci
        sc  = cfg['signals'][sig_key]
        mat, _ = extract_trials(sess['sigs'][sig_key], fip_t, events, PRE, POST, fs)
        vc = float(np.nanpercentile(np.abs(mat), 98))

        im = axes[0, col].imshow(mat, aspect='auto', cmap='viridis',
                                  interpolation='none',
                                  vmin=-vc, vmax=vc, extent=ext, origin='upper')
        axes[0, col].axvline(0, color='white', lw=1, ls='--')
        axes[0, col].set_title(sc['label'], fontsize=13)
        axes[0, col].set_xlabel(f'Time from {ev_label.lower()} (s)', fontsize=14)
        axes[0, col].set_ylabel('Trial', fontsize=14)
        axes[0, col].tick_params(labelsize=12)
        cb = plt.colorbar(im, ax=axes[0, col], label='z-score', fraction=0.05, pad=0.02)
        cb.set_label('z-score', fontsize=12)
        cb.ax.tick_params(labelsize=11)

        m = mat.mean(0); s = mat.std(0) / np.sqrt(len(mat))
        axes[1, col].plot(t_ax, m, color=sc['color'], lw=1.5)
        axes[1, col].fill_between(t_ax, m - s, m + s, color=sc['color'], alpha=0.25)
        axes[1, col].axvline(0, color='k', lw=0.8, ls='--')
        axes[1, col].axhline(0, color='#999', lw=0.4, ls=':')
        axes[1, col].set_xlabel(f'Time from {ev_label.lower()} (s)', fontsize=14)
        axes[1, col].set_ylabel('z-score', fontsize=14)
        axes[1, col].tick_params(labelsize=12)

    plt.tight_layout()
    fname = os.path.join(SAVE_DIR, f"{sess['animal_id_safe']}_threataligned_heatmap")
    plt.savefig(fname + '.pdf', bbox_inches='tight')
    plt.savefig(fname + '.png', dpi=200, bbox_inches='tight')
    plt.show(); print('저장:', fname)